In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "transformers==4.52.4",
    "accelerate==1.7.0",
    "datasets==3.6.0",
    "evaluate==0.4.3",
    "tokenizers==0.21.1"
])


0

In [1]:
import sys
import transformers
import accelerate
import datasets
import torch

print("Python executable:", sys.executable)
print("Transformers:", transformers.__version__, transformers.__file__)
print("Accelerate:", accelerate.__version__, accelerate.__file__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)

Python executable: /usr/bin/python3
Transformers: 4.52.4 /usr/local/lib/python3.13/dist-packages/transformers/__init__.py
Accelerate: 1.7.0 /usr/local/lib/python3.13/dist-packages/accelerate/__init__.py
Datasets: 3.6.0
Torch: 2.7.1+cu126


In [2]:
# =========================================================
# 1. IMPORTS
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    RobertaModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)

# =========================================================
# 2. GOOGLE DRIVE AND LOG PATHS
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

BASE_LOG_DIR = "/content/drive/MyDrive/RoBERTa_Hierarchical_Logs"
LOG_FILE_STEP1 = os.path.join(BASE_LOG_DIR, "RoBERTa_Hierarchical_Step1.csv")
LOG_FILE_STEP2 = os.path.join(BASE_LOG_DIR, "RoBERTa_Hierarchical_Step2.csv")
STEP1_SENTENCE_DIR = os.path.join(BASE_LOG_DIR, "Step1_Sentence_Logs")
STEP2_SENTENCE_DIR = os.path.join(BASE_LOG_DIR, "Step2_Combined_Sentence_Logs")

for directory in [BASE_LOG_DIR, STEP1_SENTENCE_DIR, STEP2_SENTENCE_DIR]:
    os.makedirs(directory, exist_ok=True)

# Remove old sentence logs from previous runs
for directory in [STEP1_SENTENCE_DIR, STEP2_SENTENCE_DIR]:
    for file_name in os.listdir(directory):
        if file_name.endswith(".csv"):
            os.remove(os.path.join(directory, file_name))

with open(LOG_FILE_STEP1, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["model", "roberta-base"])
    writer.writerow(["step", "STEP 1"])
    writer.writerow(["learning_rate", 2e-5])
    writer.writerow(["train_batch_size", 8])
    writer.writerow(["eval_batch_size", 8])
    writer.writerow(["epochs", 3])
    writer.writerow([])

with open(LOG_FILE_STEP2, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["model", "roberta-base"])
    writer.writerow(["step", "STEP 2"])
    writer.writerow(["learning_rate", 2e-5])
    writer.writerow(["train_batch_size", 8])
    writer.writerow(["eval_batch_size", 8])
    writer.writerow(["epochs", 3])
    writer.writerow([])

print("Step 1 results file:", LOG_FILE_STEP1)
print("Step 2 results file:", LOG_FILE_STEP2)
print("Step 1 sentence directory:", STEP1_SENTENCE_DIR)
print("Step 2 sentence directory:", STEP2_SENTENCE_DIR)

# =========================================================
# 3. SEED AND DEVICE
# =========================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================================================
# 4. LOAD DATASET
# =========================================================
print("\nLoading BRIGHTER dataset...")

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("Original sizes:")
print({
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df)
})

# =========================================================
# 5. LABEL CONFIGURATION
# =========================================================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
INTENSITY_COLUMNS = [f"{emotion}_intensity" for emotion in EMOTIONS]
LEVELS = [1, 2, 3]
LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]

print("\nEmotion labels:")
print(EMOTIONS)

print("\nCombined emotion-intensity labels:")
print(LABELS)

# =========================================================
# 6. PREPARE TWO-STEP DATA
# =========================================================
def prepare_two_step_data(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_intensity"] = df[emotion].astype(int)

    for emotion in EMOTIONS:
        df[emotion] = (df[f"{emotion}_intensity"] > 0).astype(int)

    return df

train_two = prepare_two_step_data(train_df)
val_two = prepare_two_step_data(val_df)
test_two = prepare_two_step_data(test_df)

# =========================================================
# 7. COMBINE AND REDISTRIBUTE: 70 / 20 / 10
# =========================================================
full_df = pd.concat(
    [train_two, val_two, test_two],
    ignore_index=True
)

full_df = full_df[
    ["text"] + EMOTIONS + INTENSITY_COLUMNS
]

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

number_of_rows = len(full_df)
train_end = int(0.7 * number_of_rows)
val_end = int(0.9 * number_of_rows)

train_split = full_df[:train_end].reset_index(drop=True)
val_split = full_df[train_end:val_end].reset_index(drop=True)
test_split = full_df[val_end:].reset_index(drop=True)

print("\nRedistributed split sizes:")
print({
    "train": len(train_split),
    "val": len(val_split),
    "test": len(test_split)
})

print("\nSample data:")
print(train_split.head())

test_texts = test_split["text"].tolist()

# =========================================================
# 8. STEP 1 DATA: EMOTION PRESENCE
# =========================================================
train_step1_df = train_split[["text"] + EMOTIONS].copy()
val_step1_df = val_split[["text"] + EMOTIONS].copy()
test_step1_df = test_split[["text"] + EMOTIONS].copy()

# =========================================================
# 9. STEP 2 DATA: INTENSITY PREDICTION
# =========================================================
train_step2_df = train_split[["text"] + INTENSITY_COLUMNS].copy()
val_step2_df = val_split[["text"] + INTENSITY_COLUMNS].copy()
test_step2_df = test_split[["text"] + INTENSITY_COLUMNS].copy()

# =========================================================
# 10. CONVERT TO HUGGING FACE DATASETS
# =========================================================
train_step1_ds = Dataset.from_pandas(train_step1_df, preserve_index=False)
val_step1_ds = Dataset.from_pandas(val_step1_df, preserve_index=False)
test_step1_ds = Dataset.from_pandas(test_step1_df, preserve_index=False)

train_step2_ds = Dataset.from_pandas(train_step2_df, preserve_index=False)
val_step2_ds = Dataset.from_pandas(val_step2_df, preserve_index=False)
test_step2_ds = Dataset.from_pandas(test_step2_df, preserve_index=False)

# =========================================================
# 11. TOKENIZATION
# =========================================================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_step1_ds = train_step1_ds.map(tokenize_function, batched=True)
val_step1_ds = val_step1_ds.map(tokenize_function, batched=True)
test_step1_ds = test_step1_ds.map(tokenize_function, batched=True)

train_step2_ds = train_step2_ds.map(tokenize_function, batched=True)
val_step2_ds = val_step2_ds.map(tokenize_function, batched=True)
test_step2_ds = test_step2_ds.map(tokenize_function, batched=True)

# =========================================================
# 12. ADD LABEL VECTORS
# =========================================================
def add_step1_labels(example):
    example["labels"] = [
        float(example[emotion])
        for emotion in EMOTIONS
    ]
    return example

def add_step2_labels(example):
    example["labels"] = [
        int(example[column])
        for column in INTENSITY_COLUMNS
    ]
    return example

train_step1_ds = train_step1_ds.map(add_step1_labels)
val_step1_ds = val_step1_ds.map(add_step1_labels)
test_step1_ds = test_step1_ds.map(add_step1_labels)

train_step2_ds = train_step2_ds.map(add_step2_labels)
val_step2_ds = val_step2_ds.map(add_step2_labels)
test_step2_ds = test_step2_ds.map(add_step2_labels)

# =========================================================
# 13. FORMAT DATASETS
# =========================================================
dataset_columns = [
    "input_ids",
    "attention_mask",
    "labels"
]

for dataset in [
    train_step1_ds,
    val_step1_ds,
    test_step1_ds,
    train_step2_ds,
    val_step2_ds,
    test_step2_ds
]:
    dataset.set_format(
        type="torch",
        columns=dataset_columns
    )

# =========================================================
# 14. STEP 1 METRICS
# =========================================================
def compute_metrics_step1(eval_pred):
    logits, labels = eval_pred

    probabilities = 1 / (1 + np.exp(-logits))
    predictions = (probabilities >= 0.5).astype(int)

    f1_macro = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        predictions,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for class_index in range(labels.shape[1]):
        true_column = labels[:, class_index]
        probability_column = probabilities[:, class_index]

        if (
            np.std(true_column) == 0
            or np.std(probability_column) == 0
        ):
            pearsons.append(0.0)

        else:
            correlation, _ = pearsonr(
                true_column,
                probability_column
            )

            pearsons.append(
                0.0
                if np.isnan(correlation)
                else float(correlation)
            )

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": float(np.mean(pearsons))
    }

# =========================================================
# 15. STEP 2 MODEL CLASS
# =========================================================
class RobertaStep2IntensityModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.roberta = RobertaModel.from_pretrained(
            "roberta-base"
        )

        hidden_size = self.roberta.config.hidden_size

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(
            hidden_size,
            len(EMOTIONS) * 4
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None
    ):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)

        logits = logits.view(
            -1,
            len(EMOTIONS),
            4
        )

        loss = None

        if labels is not None:
            loss_function = nn.CrossEntropyLoss()

            loss = loss_function(
                logits.view(-1, 4),
                labels.long().view(-1)
            )

        return {
            "loss": loss,
            "logits": logits
        }

# =========================================================
# 16. STEP 2 METRICS
# =========================================================
def compute_metrics_step2(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    true_flat = labels.reshape(-1)
    predicted_flat = predictions.reshape(-1)

    f1_macro = f1_score(
        true_flat,
        predicted_flat,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        true_flat,
        predicted_flat,
        average="micro",
        zero_division=0
    )

    if (
        np.std(true_flat) == 0
        or np.std(predicted_flat) == 0
    ):
        pearson_mean = 0.0

    else:
        correlation, _ = pearsonr(
            true_flat,
            predicted_flat
        )

        pearson_mean = (
            0.0
            if np.isnan(correlation)
            else float(correlation)
        )

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }

# =========================================================
# 17. SENTENCE LOGGING HELPERS
# =========================================================
def clean_number(value):
    value = round(float(value), 4)

    if value == 0:
        return 0

    if value == 1:
        return 1

    return value

def labels_to_text(
    binary_labels,
    label_names
):
    selected_labels = [
        label_names[index]
        for index, value in enumerate(binary_labels)
        if int(value) == 1
    ]

    if not selected_labels:
        return "No Emotion"

    return ", ".join(selected_labels)

# =========================================================
# 18. STEP 1 CALLBACK
# =========================================================
class SaveMetricsCallbackStep1(TrainerCallback):

    def __init__(
        self,
        file_path,
        sentence_log_dir,
        test_dataset,
        test_texts
    ):
        self.file_path = file_path
        self.sentence_log_dir = sentence_log_dir
        self.test_dataset = test_dataset
        self.test_texts = test_texts

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False
        self.processed_epochs = set()

        self.last_sentence_epoch = None
        self.last_true_labels = None
        self.last_predicted_labels = None
        self.last_probabilities = None

        self.epoch_list = []
        self.train_loss_list = []
        self.val_loss_list = []
        self.val_f1_macro_list = []
        self.val_f1_micro_list = []
        self.val_pearson_mean_list = []
        self.test_loss_list = []
        self.test_f1_macro_list = []
        self.test_f1_micro_list = []
        self.test_pearson_mean_list = []

    def on_log(
        self,
        args,
        state,
        control,
        logs=None,
        **kwargs
    ):
        if (
            logs is not None
            and "loss" in logs
            and "eval_loss" not in logs
        ):
            self.current_train_loss = float(
                logs["loss"]
            )

    def save_sentence_log(
        self,
        epoch,
        true_labels,
        predicted_labels,
        probabilities
    ):
        output_path = os.path.join(
            self.sentence_log_dir,
            f"RoBERTa_Step1_Epoch_{epoch}_Test_Sentences.csv"
        )

        rows = []

        for sentence_index, sentence in enumerate(self.test_texts):
            row = {
                "epoch": epoch,
                "sentence_id": sentence_index,
                "sentence": sentence,
                "true_emotions": labels_to_text(
                    true_labels[sentence_index],
                    EMOTIONS
                ),
                "predicted_emotions": labels_to_text(
                    predicted_labels[sentence_index],
                    EMOTIONS
                )
            }

            for emotion_index, emotion in enumerate(EMOTIONS):
                row[f"true_{emotion}"] = int(
                    true_labels[
                        sentence_index,
                        emotion_index
                    ]
                )

                row[f"predicted_{emotion}"] = int(
                    predicted_labels[
                        sentence_index,
                        emotion_index
                    ]
                )

                row[f"prob_{emotion}"] = clean_number(
                    probabilities[
                        sentence_index,
                        emotion_index
                    ]
                )

            rows.append(row)

        pd.DataFrame(rows).to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    def on_evaluate(
        self,
        args,
        state,
        control,
        metrics=None,
        **kwargs
    ):
        if (
            self._inside_eval
            or metrics is None
            or "eval_loss" not in metrics
        ):
            return

        epoch = int(
            round(
                float(
                    metrics.get(
                        "epoch",
                        state.epoch
                    )
                )
            )
        )

        if epoch in self.processed_epochs:
            return

        self._inside_eval = True

        try:
            train_loss = (
                self.current_train_loss
                if self.current_train_loss is not None
                else ""
            )

            val_loss = float(
                metrics.get(
                    "eval_loss",
                    0.0
                )
            )

            val_f1_macro = float(
                metrics.get(
                    "eval_f1_macro",
                    0.0
                )
            )

            val_f1_micro = float(
                metrics.get(
                    "eval_f1_micro",
                    0.0
                )
            )

            val_pearson_mean = float(
                metrics.get(
                    "eval_pearson_mean",
                    0.0
                )
            )

            prediction_output = self.trainer_ref.predict(
                self.test_dataset,
                metric_key_prefix="test"
            )

            test_metrics = prediction_output.metrics

            test_loss = float(
                test_metrics.get(
                    "test_loss",
                    0.0
                )
            )

            test_f1_macro = float(
                test_metrics.get(
                    "test_f1_macro",
                    0.0
                )
            )

            test_f1_micro = float(
                test_metrics.get(
                    "test_f1_micro",
                    0.0
                )
            )

            test_pearson_mean = float(
                test_metrics.get(
                    "test_pearson_mean",
                    0.0
                )
            )

            logits = prediction_output.predictions
            true_labels = prediction_output.label_ids.astype(int)

            probabilities = 1 / (
                1 + np.exp(-logits)
            )

            predicted_labels = (
                probabilities >= 0.5
            ).astype(int)

            self.epoch_list.append(epoch)
            self.train_loss_list.append(train_loss)
            self.val_loss_list.append(val_loss)
            self.val_f1_macro_list.append(val_f1_macro)
            self.val_f1_micro_list.append(val_f1_micro)
            self.val_pearson_mean_list.append(val_pearson_mean)
            self.test_loss_list.append(test_loss)
            self.test_f1_macro_list.append(test_f1_macro)
            self.test_f1_micro_list.append(test_f1_micro)
            self.test_pearson_mean_list.append(test_pearson_mean)

            report_dict = classification_report(
                true_labels,
                predicted_labels,
                target_names=EMOTIONS,
                zero_division=0,
                output_dict=True
            )

            with open(
                self.file_path,
                "a",
                newline="",
                encoding="utf-8"
            ) as file:
                writer = csv.writer(file)

                writer.writerow([])
                writer.writerow([f"EPOCH {epoch}"])

                writer.writerow([
                    "epoch",
                    "train_loss",
                    "val_loss",
                    "test_loss",
                    "val_f1_macro",
                    "val_f1_micro",
                    "test_f1_macro",
                    "test_f1_micro",
                    "val_pearson_mean",
                    "test_pearson_mean"
                ])

                for index in range(len(self.epoch_list)):
                    writer.writerow([
                        self.epoch_list[index],
                        self.train_loss_list[index],
                        self.val_loss_list[index],
                        self.test_loss_list[index],
                        self.val_f1_macro_list[index],
                        self.val_f1_micro_list[index],
                        self.test_f1_macro_list[index],
                        self.test_f1_micro_list[index],
                        self.val_pearson_mean_list[index],
                        self.test_pearson_mean_list[index]
                    ])

                writer.writerow([])
                writer.writerow([
                    f"FINAL TEST SCORES AFTER EPOCH {epoch}"
                ])

                writer.writerow([
                    "metric",
                    "value"
                ])

                writer.writerow([
                    "test_loss",
                    test_loss
                ])

                writer.writerow([
                    "test_f1_macro",
                    test_f1_macro
                ])

                writer.writerow([
                    "test_f1_micro",
                    test_f1_micro
                ])

                writer.writerow([
                    "test_pearson_mean",
                    test_pearson_mean
                ])

                writer.writerow([])
                writer.writerow([
                    f"CLASSWISE RESULTS AFTER EPOCH {epoch}"
                ])

                writer.writerow([
                    "class",
                    "precision",
                    "recall",
                    "f1_score",
                    "correct_predictions",
                    "support"
                ])

                for class_index, class_name in enumerate(EMOTIONS):
                    class_result = report_dict.get(
                        class_name,
                        {}
                    )

                    correct_predictions = int(
                        np.sum(
                            (
                                true_labels[
                                    :,
                                    class_index
                                ] == 1
                            )
                            &
                            (
                                predicted_labels[
                                    :,
                                    class_index
                                ] == 1
                            )
                        )
                    )

                    writer.writerow([
                        class_name,
                        class_result.get(
                            "precision",
                            ""
                        ),
                        class_result.get(
                            "recall",
                            ""
                        ),
                        class_result.get(
                            "f1-score",
                            ""
                        ),
                        correct_predictions,
                        class_result.get(
                            "support",
                            ""
                        )
                    ])

            # Store latest test predictions only
            self.last_sentence_epoch = epoch
            self.last_true_labels = true_labels.copy()
            self.last_predicted_labels = predicted_labels.copy()
            self.last_probabilities = probabilities.copy()

            self.processed_epochs.add(epoch)

            print(
                f"\nStep 1 epoch {epoch} metrics saved."
            )

        finally:
            self._inside_eval = False

    def on_train_end(
        self,
        args,
        state,
        control,
        **kwargs
    ):
        if self.last_sentence_epoch is None:
            return

        sentence_path = self.save_sentence_log(
            epoch=self.last_sentence_epoch,
            true_labels=self.last_true_labels,
            predicted_labels=self.last_predicted_labels,
            probabilities=self.last_probabilities
        )

        print(
            "\nStep 1 final test sentence log saved "
            f"from epoch {self.last_sentence_epoch}."
        )

        print(
            "Sentence file:",
            sentence_path
        )

# =========================================================
# 19. STEP 2 CALLBACK
# =========================================================
class SaveMetricsCallbackStep2(TrainerCallback):

    def __init__(
        self,
        file_path,
        sentence_log_dir,
        step1_trainer,
        test_step1_dataset,
        test_step2_dataset,
        test_texts
    ):
        self.file_path = file_path
        self.sentence_log_dir = sentence_log_dir
        self.step1_trainer = step1_trainer
        self.test_step1_dataset = test_step1_dataset
        self.test_step2_dataset = test_step2_dataset
        self.test_texts = test_texts

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False
        self.processed_epochs = set()

        self.last_sentence_epoch = None
        self.last_prediction_data = None

        self.epoch_list = []
        self.train_loss_list = []
        self.val_loss_list = []
        self.val_f1_macro_list = []
        self.val_f1_micro_list = []
        self.val_pearson_mean_list = []
        self.test_loss_list = []
        self.test_f1_macro_list = []
        self.test_f1_micro_list = []
        self.test_pearson_mean_list = []

    def on_log(
        self,
        args,
        state,
        control,
        logs=None,
        **kwargs
    ):
        if (
            logs is not None
            and "loss" in logs
            and "eval_loss" not in logs
        ):
            self.current_train_loss = float(
                logs["loss"]
            )

    def make_final_hierarchical_predictions(self):
        step1_output = self.step1_trainer.predict(
            self.test_step1_dataset,
            metric_key_prefix="step1_test"
        )

        true_emotions = step1_output.label_ids.astype(int)

        step1_probabilities = 1 / (
            1 + np.exp(
                -step1_output.predictions
            )
        )

        predicted_emotions = (
            step1_probabilities >= 0.5
        ).astype(int)

        step2_output = self.trainer_ref.predict(
            self.test_step2_dataset,
            metric_key_prefix="test"
        )

        step2_logits = step2_output.predictions
        true_intensities = step2_output.label_ids.astype(int)

        test_loss = float(
            step2_output.metrics.get(
                "test_loss",
                0.0
            )
        )

        step2_probabilities = torch.softmax(
            torch.tensor(step2_logits),
            dim=-1
        ).numpy()

        step2_predicted_intensities = np.argmax(
            step2_probabilities,
            axis=-1
        )

        final_predicted_intensities = (
            step2_predicted_intensities
            * predicted_emotions
        )

        true_binary_15 = np.zeros(
            (
                true_intensities.shape[0],
                len(LABELS)
            ),
            dtype=int
        )

        predicted_binary_15 = np.zeros(
            (
                true_intensities.shape[0],
                len(LABELS)
            ),
            dtype=int
        )

        combined_probabilities = np.zeros(
            (
                true_intensities.shape[0],
                len(LABELS)
            ),
            dtype=float
        )

        for emotion_index, emotion in enumerate(EMOTIONS):
            for level in LEVELS:
                class_index = (
                    emotion_index * 3
                    + level
                    - 1
                )

                true_binary_15[
                    :,
                    class_index
                ] = (
                    true_intensities[
                        :,
                        emotion_index
                    ] == level
                ).astype(int)

                predicted_binary_15[
                    :,
                    class_index
                ] = (
                    final_predicted_intensities[
                        :,
                        emotion_index
                    ] == level
                ).astype(int)

                combined_probabilities[
                    :,
                    class_index
                ] = (
                    step1_probabilities[
                        :,
                        emotion_index
                    ]
                    *
                    step2_probabilities[
                        :,
                        emotion_index,
                        level
                    ]
                )

        return {
            "true_emotions": true_emotions,
            "predicted_emotions": predicted_emotions,
            "step1_probabilities": step1_probabilities,
            "true_intensities": true_intensities,
            "step2_probabilities": step2_probabilities,
            "step2_predicted_intensities": step2_predicted_intensities,
            "final_predicted_intensities": final_predicted_intensities,
            "true_binary_15": true_binary_15,
            "predicted_binary_15": predicted_binary_15,
            "combined_probabilities": combined_probabilities,
            "test_loss": test_loss
        }

    def save_combined_sentence_log(
        self,
        epoch,
        prediction_data
    ):
        output_path = os.path.join(
            self.sentence_log_dir,
            f"RoBERTa_Step2_Epoch_{epoch}_Combined_Test_Sentences.csv"
        )

        rows = []

        for sentence_index, sentence in enumerate(self.test_texts):
            row = {
                "epoch": epoch,
                "sentence_id": sentence_index,
                "sentence": sentence,
                "true_emotions": labels_to_text(
                    prediction_data[
                        "true_emotions"
                    ][sentence_index],
                    EMOTIONS
                ),
                "predicted_emotions": labels_to_text(
                    prediction_data[
                        "predicted_emotions"
                    ][sentence_index],
                    EMOTIONS
                ),
                "true_combined_labels": labels_to_text(
                    prediction_data[
                        "true_binary_15"
                    ][sentence_index],
                    LABELS
                ),
                "predicted_combined_labels": labels_to_text(
                    prediction_data[
                        "predicted_binary_15"
                    ][sentence_index],
                    LABELS
                )
            }

            for emotion_index, emotion in enumerate(EMOTIONS):
                row[f"true_{emotion}"] = int(
                    prediction_data[
                        "true_emotions"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[f"predicted_{emotion}"] = int(
                    prediction_data[
                        "predicted_emotions"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[f"step1_prob_{emotion}"] = clean_number(
                    prediction_data[
                        "step1_probabilities"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[f"true_{emotion}_intensity"] = int(
                    prediction_data[
                        "true_intensities"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[f"step2_predicted_{emotion}_intensity"] = int(
                    prediction_data[
                        "step2_predicted_intensities"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[f"final_predicted_{emotion}_intensity"] = int(
                    prediction_data[
                        "final_predicted_intensities"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                for level in range(4):
                    row[
                        f"step2_prob_{emotion}_intensity_{level}"
                    ] = clean_number(
                        prediction_data[
                            "step2_probabilities"
                        ][
                            sentence_index,
                            emotion_index,
                            level
                        ]
                    )

            for class_index, class_name in enumerate(LABELS):
                row[
                    f"combined_prob_{class_name}"
                ] = clean_number(
                    prediction_data[
                        "combined_probabilities"
                    ][
                        sentence_index,
                        class_index
                    ]
                )

            rows.append(row)

        pd.DataFrame(rows).to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    def on_evaluate(
        self,
        args,
        state,
        control,
        metrics=None,
        **kwargs
    ):
        if (
            self._inside_eval
            or metrics is None
            or "eval_loss" not in metrics
        ):
            return

        epoch = int(
            round(
                float(
                    metrics.get(
                        "epoch",
                        state.epoch
                    )
                )
            )
        )

        if epoch in self.processed_epochs:
            return

        self._inside_eval = True

        try:
            train_loss = (
                self.current_train_loss
                if self.current_train_loss is not None
                else ""
            )

            val_loss = float(
                metrics.get(
                    "eval_loss",
                    0.0
                )
            )

            val_f1_macro = float(
                metrics.get(
                    "eval_f1_macro",
                    0.0
                )
            )

            val_f1_micro = float(
                metrics.get(
                    "eval_f1_micro",
                    0.0
                )
            )

            val_pearson_mean = float(
                metrics.get(
                    "eval_pearson_mean",
                    0.0
                )
            )

            prediction_data = (
                self.make_final_hierarchical_predictions()
            )

            true_binary_15 = prediction_data[
                "true_binary_15"
            ]

            predicted_binary_15 = prediction_data[
                "predicted_binary_15"
            ]

            combined_probabilities = prediction_data[
                "combined_probabilities"
            ]

            test_loss = prediction_data[
                "test_loss"
            ]

            test_f1_macro = f1_score(
                true_binary_15,
                predicted_binary_15,
                average="macro",
                zero_division=0
            )

            test_f1_micro = f1_score(
                true_binary_15,
                predicted_binary_15,
                average="micro",
                zero_division=0
            )

            pearsons = []

            for class_index in range(len(LABELS)):
                true_column = true_binary_15[
                    :,
                    class_index
                ]

                probability_column = combined_probabilities[
                    :,
                    class_index
                ]

                if (
                    np.std(true_column) == 0
                    or np.std(probability_column) == 0
                ):
                    pearsons.append(0.0)

                else:
                    correlation, _ = pearsonr(
                        true_column,
                        probability_column
                    )

                    pearsons.append(
                        0.0
                        if np.isnan(correlation)
                        else float(correlation)
                    )

            test_pearson_mean = float(
                np.mean(pearsons)
            )

            self.epoch_list.append(epoch)
            self.train_loss_list.append(train_loss)
            self.val_loss_list.append(val_loss)
            self.val_f1_macro_list.append(val_f1_macro)
            self.val_f1_micro_list.append(val_f1_micro)
            self.val_pearson_mean_list.append(val_pearson_mean)
            self.test_loss_list.append(test_loss)
            self.test_f1_macro_list.append(test_f1_macro)
            self.test_f1_micro_list.append(test_f1_micro)
            self.test_pearson_mean_list.append(test_pearson_mean)

            report_dict = classification_report(
                true_binary_15,
                predicted_binary_15,
                target_names=LABELS,
                zero_division=0,
                output_dict=True
            )

            with open(
                self.file_path,
                "a",
                newline="",
                encoding="utf-8"
            ) as file:
                writer = csv.writer(file)

                writer.writerow([])
                writer.writerow([f"EPOCH {epoch}"])

                writer.writerow([
                    "epoch",
                    "train_loss",
                    "val_loss",
                    "test_loss",
                    "val_f1_macro",
                    "val_f1_micro",
                    "test_f1_macro",
                    "test_f1_micro",
                    "val_pearson_mean",
                    "test_pearson_mean"
                ])

                for index in range(len(self.epoch_list)):
                    writer.writerow([
                        self.epoch_list[index],
                        self.train_loss_list[index],
                        self.val_loss_list[index],
                        self.test_loss_list[index],
                        self.val_f1_macro_list[index],
                        self.val_f1_micro_list[index],
                        self.test_f1_macro_list[index],
                        self.test_f1_micro_list[index],
                        self.val_pearson_mean_list[index],
                        self.test_pearson_mean_list[index]
                    ])

                writer.writerow([])
                writer.writerow([
                    f"FINAL TEST SCORES AFTER EPOCH {epoch}"
                ])

                writer.writerow([
                    "metric",
                    "value"
                ])

                writer.writerow([
                    "test_loss",
                    test_loss
                ])

                writer.writerow([
                    "test_f1_macro",
                    test_f1_macro
                ])

                writer.writerow([
                    "test_f1_micro",
                    test_f1_micro
                ])

                writer.writerow([
                    "test_pearson_mean",
                    test_pearson_mean
                ])

                writer.writerow([])
                writer.writerow([
                    f"CLASSWISE RESULTS AFTER EPOCH {epoch}"
                ])

                writer.writerow([
                    "class",
                    "precision",
                    "recall",
                    "f1_score",
                    "correct_predictions",
                    "support"
                ])

                for class_index, class_name in enumerate(LABELS):
                    class_result = report_dict.get(
                        class_name,
                        {}
                    )

                    correct_predictions = int(
                        np.sum(
                            (
                                true_binary_15[
                                    :,
                                    class_index
                                ] == 1
                            )
                            &
                            (
                                predicted_binary_15[
                                    :,
                                    class_index
                                ] == 1
                            )
                        )
                    )

                    writer.writerow([
                        class_name,
                        class_result.get(
                            "precision",
                            ""
                        ),
                        class_result.get(
                            "recall",
                            ""
                        ),
                        class_result.get(
                            "f1-score",
                            ""
                        ),
                        correct_predictions,
                        class_result.get(
                            "support",
                            ""
                        )
                    ])

            # Keep only latest test predictions
            self.last_sentence_epoch = epoch
            self.last_prediction_data = prediction_data

            self.processed_epochs.add(epoch)

            print(
                f"\nStep 2 epoch {epoch} metrics saved."
            )

        finally:
            self._inside_eval = False

    def on_train_end(
        self,
        args,
        state,
        control,
        **kwargs
    ):
        if self.last_sentence_epoch is None:
            return

        sentence_path = self.save_combined_sentence_log(
            epoch=self.last_sentence_epoch,
            prediction_data=self.last_prediction_data
        )

        print(
            "\nStep 2 final combined test sentence log "
            f"saved from epoch {self.last_sentence_epoch}."
        )

        print(
            "Combined sentence file:",
            sentence_path
        )

# =========================================================
# 20. STEP 1 MODEL
# =========================================================
model_step1 = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(EMOTIONS),
    problem_type="multi_label_classification"
)

# =========================================================
# 21. STEP 1 TRAINING ARGUMENTS
# =========================================================
training_args_step1 = TrainingArguments(
    output_dir="/content/roberta_step1_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

# =========================================================
# 22. STEP 1 TRAINER
# =========================================================
callback_step1 = SaveMetricsCallbackStep1(
    file_path=LOG_FILE_STEP1,
    sentence_log_dir=STEP1_SENTENCE_DIR,
    test_dataset=test_step1_ds,
    test_texts=test_texts
)

trainer_step1 = Trainer(
    model=model_step1,
    args=training_args_step1,
    train_dataset=train_step1_ds,
    eval_dataset=val_step1_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_step1,
    callbacks=[
        callback_step1,
        EarlyStoppingCallback(
            early_stopping_patience=1,
            early_stopping_threshold=0.0
        )
    ]
)

callback_step1.trainer_ref = trainer_step1

# =========================================================
# 23. TRAIN STEP 1
# =========================================================
print("\nStarting Step 1: Emotion Detection")

step1_start_time = time.time()
trainer_step1.train()
step1_end_time = time.time()

print(
    f"Step 1 training time: "
    f"{step1_end_time - step1_start_time:.1f} seconds"
)

print(
    "Step 1 result file:",
    LOG_FILE_STEP1
)

print(
    "Step 1 final test sentence directory:",
    STEP1_SENTENCE_DIR
)

# =========================================================
# 24. STEP 2 MODEL INSTANCE
# =========================================================
model_step2 = RobertaStep2IntensityModel()

# =========================================================
# 25. STEP 2 TRAINING ARGUMENTS
# =========================================================
training_args_step2 = TrainingArguments(
    output_dir="/content/roberta_step2_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

# =========================================================
# 26. STEP 2 TRAINER
# =========================================================
callback_step2 = SaveMetricsCallbackStep2(
    file_path=LOG_FILE_STEP2,
    sentence_log_dir=STEP2_SENTENCE_DIR,
    step1_trainer=trainer_step1,
    test_step1_dataset=test_step1_ds,
    test_step2_dataset=test_step2_ds,
    test_texts=test_texts
)

trainer_step2 = Trainer(
    model=model_step2,
    args=training_args_step2,
    train_dataset=train_step2_ds,
    eval_dataset=val_step2_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_step2,
    callbacks=[
        callback_step2,
        EarlyStoppingCallback(
            early_stopping_patience=1,
            early_stopping_threshold=0.0
        )
    ]
)

callback_step2.trainer_ref = trainer_step2

# =========================================================
# 27. TRAIN STEP 2
# =========================================================
print("\nStarting Step 2: Intensity Classification")

step2_start_time = time.time()
trainer_step2.train()
step2_end_time = time.time()

print(
    f"Step 2 training time: "
    f"{step2_end_time - step2_start_time:.1f} seconds"
)

print(
    "Step 2 result file:",
    LOG_FILE_STEP2
)

print(
    "Step 2 final combined test sentence directory:",
    STEP2_SENTENCE_DIR
)

# =========================================================
# 28. DISPLAY CREATED SENTENCE FILES
# =========================================================
print("\nStep 1 final test sentence file:")

for file_name in sorted(os.listdir(STEP1_SENTENCE_DIR)):
    if file_name.endswith(".csv"):
        print(
            os.path.join(
                STEP1_SENTENCE_DIR,
                file_name
            )
        )

print("\nStep 2 final combined test sentence file:")

for file_name in sorted(os.listdir(STEP2_SENTENCE_DIR)):
    if file_name.endswith(".csv"):
        print(
            os.path.join(
                STEP2_SENTENCE_DIR,
                file_name
            )
        )




Mounted at /content/drive
Step 1 results file: /content/drive/MyDrive/RoBERTa_Hierarchical_Logs/RoBERTa_Hierarchical_Step1.csv
Step 2 results file: /content/drive/MyDrive/RoBERTa_Hierarchical_Logs/RoBERTa_Hierarchical_Step2.csv
Step 1 sentence directory: /content/drive/MyDrive/RoBERTa_Hierarchical_Logs/Step1_Sentence_Logs
Step 2 sentence directory: /content/drive/MyDrive/RoBERTa_Hierarchical_Logs/Step2_Combined_Sentence_Logs
Using device: cuda

Loading BRIGHTER dataset...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

eng/train-00000-of-00001.parquet:   0%|          | 0.00/179k [00:00<?, ?B/s]

eng/dev-00000-of-00001.parquet:   0%|          | 0.00/11.8k [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Pearson Mean
1,0.746200,0.605577,0.499129,0.749291,0.666131
2,0.546700,0.562228,0.572285,0.767021,0.739382
3,0.437400,0.560031,0.580323,0.773582,0.751140
4,0.365400,0.554682,0.597931,0.779433,0.761236



Step 2 epoch 1 metrics saved.



Step 2 epoch 2 metrics saved.



Step 2 epoch 3 metrics saved.



Step 2 epoch 4 metrics saved.

Step 2 final combined test sentence log saved from epoch 4.
Combined sentence file: /content/drive/MyDrive/RoBERTa_Hierarchical_Logs/Step2_Combined_Sentence_Logs/RoBERTa_Step2_Epoch_4_Combined_Test_Sentences.csv
Step 2 training time: 331.6 seconds
Step 2 result file: /content/drive/MyDrive/RoBERTa_Hierarchical_Logs/RoBERTa_Hierarchical_Step2.csv
Step 2 final combined test sentence directory: /content/drive/MyDrive/RoBERTa_Hierarchical_Logs/Step2_Combined_Sentence_Logs

Step 1 final test sentence file:
/content/drive/MyDrive/RoBERTa_Hierarchical_Logs/Step1_Sentence_Logs/RoBERTa_Step1_Epoch_3_Test_Sentences.csv

Step 2 final combined test sentence file:
/content/drive/MyDrive/RoBERTa_Hierarchical_Logs/Step2_Combined_Sentence_Logs/RoBERTa_Step2_Epoch_4_Combined_Test_Sentences.csv
